# Zero-shot DSR using DynaMix


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import sys
sys.path.append('..')

from DynaMix.src.model.dynamix import DynaMix
from DynaMix.src.model.forecaster import DynaMixForecaster
from DynaMix.src.metrics.metrics import geometrical_misalignment, temporal_misalignment, MASE
from DynaMix.src.utilities.plotting_eval import plot_3D_attractor, plot_2D_attractor, plot_TS_forecast
from DynaMix.src.utilities.utilities import load_hf_model

## Dysts Benchmarks

In [ ]:
from dysts.metrics import smape


def vpt_smape(x, xhat, threshold=30):
    """
    Find the first time index at which an array exceeds a threshold.

    Args:
        x (np.ndarray): The ground truth, a time series of shape (nt, 1).
        xhat (np.ndarray): The forecast, a time series of shape (nt, 1).
        threshold (float): The threshold to search for.

    Returns:
        int: The first time index at which the array exceeds the threshold.
    """
    arr = horizoned_smape(x, xhat)
    exceed_times = np.where(arr > threshold)[0]
    if len(exceed_times) == 0:
        tind = len(arr)
    else:
        tind = exceed_times[0]
    return tind

def horizoned_smape(x, xhat):
    """Given a horizoned forecast and ground truth, compute the SMAPE as a function of time"""
    nt = min(x.shape[0], xhat.shape[0])
    smape_vals = list()
    for i in range(1, nt+1):
        smape_vals.append(smape(x[:i], xhat[:i]))
    smape_vals = np.array(smape_vals)
    return smape_vals

In [ ]:
import glob
import os
# context_path
context_paths = sorted(glob.glob("/Users/william/program_repos/dysts_data/benchmark_results/zero-shot/trajectory_data/context_*context.npy"))
output_dir= "/Users/william/program_repos/dysts_data/benchmark_results/zero-shot/dynamix_context_512_granularity_30/"
# ground_truth_paths = sorted(glob.glob("/Users/william/program_repos/dysts_data/benchmark_results/zero-shot/trajectory_data/forecast_*.npy"))

# Load the pre-trained model
model = load_hf_model("dynamix-3d-alrnn-v1.0")
model.eval() # Set model to evaluation mode
forecaster = DynaMixForecaster(model) # Initialize the forecaster

## Store the metrics
all_all_smape_rolling = list()
all_all_vpt = list()
all_all_cdim_true = list()
all_all_cdim_pred = list()
all_all_kl_dist = list()

for context_path in context_paths[:]:
    context_traj = np.load(context_path)
    fname = os.path.split(context_path)[1]
    eq_name = fname.split("_")[1]
    print(eq_name, flush=True)

    ## Load context and make tensor
    context_traj_tensor = torch.tensor(context_traj, dtype=torch.float32)
    context_traj_tensor = torch.transpose(context_traj_tensor, 0, 1)

    ## Get ground truth
    ground_truth_path = f"/Users/william/program_repos/dysts_data/benchmark_results/zero-shot/trajectory_data/forecast_{eq_name}_granularity30_true_chronos_test.npy"
    ground_truth = np.load(ground_truth_path)
    ground_truth_tensor = torch.tensor(ground_truth, dtype=torch.float32)

    ## Make DynaMix prediction
    with torch.no_grad(): 
        reconstruction = forecaster.forecast(
            context=context_traj_tensor,
            horizon=ground_truth.shape[1], # Match the horizon of the ground truth
            standardize=True,
        )
    reconstruction = torch.transpose(reconstruction, 0, 1)


    ## Make output path and save reconstruction to it
    output_path = os.path.join(output_dir, f"dynamix_{eq_name}_large_granularity30.npy")
    np.save(output_path, reconstruction.detach().numpy(), allow_pickle=True)



    ### Error metrics
    all_traj_true = ground_truth_tensor.detach().numpy()
    all_traj_forecasts = reconstruction.detach().numpy()

    ## Compute horizoned smape
    all_smape_rolling = list()
    all_vpt = list()
    for traj_pred, traj_true in zip(all_traj_forecasts, all_traj_true):
        all_smape = [horizoned_smape(traj_true[:, i], traj_pred[:, i]) for i in range(traj_pred.shape[1])]
        all_smape_rolling.append(all_smape)
        all_vpt.append(vpt_smape(traj_pred.squeeze(), traj_true.squeeze()))
    all_smape_rolling = np.array(all_smape_rolling)
    all_vpt = np.array(all_vpt)
    ## Across replicate initial conditions, take median
    ## Across different dimensions, take mean
    averaged_smape_rolling = np.nanmean(np.copy(all_smape_rolling), axis=1)

    cdim_true, cdim_pred = gp_dim(np.vstack(all_traj_true)), gp_dim(np.vstack(all_traj_forecasts).squeeze())

    kl_dist = estimate_kl_divergence(np.vstack(all_traj_true).squeeze(), np.vstack(all_traj_forecasts).squeeze())
    if np.isinf(kl_dist):
        kl_dist = np.nan

    all_all_smape_rolling.append(averaged_smape_rolling)
    all_all_vpt.append(np.nanmean(all_vpt))
    all_all_cdim_true.append(cdim_true)
    all_all_cdim_pred.append(cdim_pred)
    all_all_kl_dist.append(kl_dist)

all_all_smape_rolling = np.array(all_all_smape_rolling)
all_all_vpt = np.array(all_all_vpt)
all_all_kl_dist = np.array(all_all_kl_dist)
from scipy.stats import spearmanr
scorr_val = spearmanr(all_all_cdim_true, all_all_cdim_pred).statistic
scorr_val_err = 1 / np.sqrt(len(all_all_cdim_true) - 1 )



Aizawa
AnishchenkoAstakhov
Arneodo
ArnoldBeltramiChildress
ArnoldWeb
AtmosphericRegime


/Users/william/program_repos/dynamix_baseline/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:193: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/Users/william/program_repos/dynamix_baseline/.venv/lib/python3.13/site-packages/numpy/_core/function_base.py:140: RuntimeWarning: invalid value encountered in subtract
  delta = np.subtract(stop, start, dtype=type(dt))


BeerRNN
BelousovZhabotinsky
BickleyJet
Blasius
BlinkingRotlet
BlinkingVortex
Bouali2
Bouali
BurkeShaw
CaTwoPlusQuasiperiodic
CaTwoPlus
CellCycle
CellularNeuralNetwork
ChenLee


/Users/william/program_repos/dynamix_baseline/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:204: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)


Chen
Chua
CircadianRhythm
CoevolvingPredatorPrey
Colpitts
Coullet
Dadras
DequanLi
DoubleGyre


/Users/william/program_repos/dynamix_baseline/.venv/lib/python3.13/site-packages/dysts/metrics.py:543: RuntimeWarning: divide by zero encountered in divide
  log_ratios = np.log(p_hat(selected_samples) / q_hat(selected_samples))
/Users/william/program_repos/dynamix_baseline/.venv/lib/python3.13/site-packages/dysts/metrics.py:543: RuntimeWarning: overflow encountered in divide
  log_ratios = np.log(p_hat(selected_samples) / q_hat(selected_samples))


DoublePendulum
Duffing
ExcitableCell
Finance
FluidTrampoline
ForcedBrusselator
ForcedFitzHughNagumo
ForcedVanDerPol
GenesioTesi
GuckenheimerHolmes
Hadley
Halvorsen
HastingsPowell
HenonHeiles
HindmarshRose
Hopfield
HyperBao
HyperCai
HyperJha
HyperLorenz
HyperPang
HyperQi
HyperRossler
HyperWang
HyperXu
HyperYan
HyperYangChen
IsothermalChemical
ItikBanksTumor
JerkCircuit
KawczynskiStrizhak
Laser
LidDrivenCavityFlow
LiuChen
Lorenz84
Lorenz96
LorenzBounded
LorenzCoupled
LorenzStenflo
Lorenz
LuChenCheng
LuChen
MacArthur
MooreSpiegel
MultiChua
NewtonLiepnik
NoseHoover
NuclearQuadrupole
OscillatingFlow
PanXuZhou
PehlivanWei
QiChen
Qi
RabinovichFabrikant
RayleighBenard
RikitakeDynamo
Rossler
Rucklidge
Sakarya
SaltonSea
SanUmSrisuchinwong
ShimizuMorioka
SprottA
SprottB
SprottC
SprottD
SprottE
SprottF
SprottG
SprottH
SprottI
SprottJ
SprottJerk
SprottK
SprottL
SprottM
SprottMore
SprottN
SprottO
SprottP
SprottQ
SprottR
SprottS
SprottTorus


In [85]:


## save to npz file
# np.savez("all_all_metrics.npz", smape_rolling=all_all_smape_rolling, vpt=all_all_vpt, kl_dist=all_all_kl_dist, corrdim_corr=scorr_val, corrdim_corr_err=scorr_val_err)

out = np.load("all_all_metrics.npz", allow_pickle=True)
smape_rolling = out["smape_rolling"]
all_vpt = out["vpt"]
kl_dist = out["kl_dist"]
corrdim_corr = out["corrdim_corr"]
corrdim_corr_err = out["corrdim_corr_err"]




In [74]:
out["smape_rolling"].shape

(113, 20, 300)